In [1]:
from dotenv import  load_dotenv , find_dotenv
import  pandas as pd
from pinecone import  Pinecone , ServerlessSpec
from sentence_transformers import SentenceTransformer
import os

C:\Users\akhil\Langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv())

True

In [3]:
model = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 240.73it/s]


In [4]:
pc = Pinecone(
    api_key=os.getenv("PINECE_API_KEY")
)

In [5]:
index_name = 'bert'
metric = "cosine"

In [6]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f'Deleted : {index_name}')

Deleted : bert


In [7]:
pc.create_index(
    name=index_name,
    dimension= model.get_embedding_dimension(),
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

Name:,bert
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://bert-rff7cv2.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [9]:
files = pd.read_csv('course_section_descriptions.csv',encoding = "ANSI")

In [10]:
files['unique_id'] = files['course_id'].astype(str)+'-'+files['section_id'].astype(str)

In [11]:
files['meta_data'] = files.apply(
    lambda courses : {
        "course_name" : courses['course_name'],
        "section_name" : courses['section_name'],
        "section_description" : courses['section_description']
    },axis=1
)

In [12]:
def create_embeddings(row):
    text = f''' {row['course_name']} {row['course_technology']} {row['course_description']}{row['section_name']} {row['section_description']} '''
    return model.encode(text)

In [13]:
files['embeddings'] = files.apply(
    lambda row : create_embeddings(row), axis=1
)

In [14]:
vectors_to_upsert = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for index, row in files.iterrows()  ]

KeyError: 'embedding'

In [ ]:
index = pc.index(index_name)
index.upsert(vectors = vectors_to_upsert)
print("Data succesfully upserted to Pinecone index")